In [5]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

PROCESSED = PROJECT_ROOT / "data" / "processed"

USER_DAY = PROCESSED / "user_day_r62_v1.parquet"

user_day_path = str(USER_DAY).replace("\\", "/")

assert USER_DAY.exists(), f"Missing: {USER_DAY}"


# =========================================================
# OFFICIAL ORIGINAL MALICIOUS USERS
# =========================================================

malicious_users = [
    "ACM2278",
    "CMP2946",
    "PLJ1771",
    "CDE1846",
    "MBG3183"
]

malicious_sql = ", ".join(
    f"'{u}'"
    for u in malicious_users
)


print("User-Day file:", USER_DAY)
print("Original malicious users excluded:", len(malicious_users))

print("\n✅ SUCCESS — Baseline audit setup is ready.")

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
User-Day file: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\user_day_r62_v1.parquet
Original malicious users excluded: 5

✅ SUCCESS — Baseline audit setup is ready.


In [7]:
# =========================================================
# CANDIDATE BASELINE CUTOFFS
# =========================================================

cutoffs = [

    "2010-07-01",
    "2010-08-01",
    "2010-09-01",
    "2010-10-01",
    "2010-11-01",
    "2010-12-01",

    "2011-01-01",
    "2011-02-01"
]


print("Candidate cutoffs:")

for cutoff in cutoffs:
    print("-", cutoff)

print(
    "\n✅ SUCCESS — Candidate cutoff dates defined."
)

Candidate cutoffs:
- 2010-07-01
- 2010-08-01
- 2010-09-01
- 2010-10-01
- 2010-11-01
- 2010-12-01
- 2011-01-01
- 2011-02-01

✅ SUCCESS — Candidate cutoff dates defined.


In [9]:
# =========================================================
# BASELINE WINDOW AUDIT
# =========================================================

con = duckdb.connect()

audit_rows = []


for cutoff in cutoffs:

    result = con.execute(f"""

        WITH user_stats AS (

            SELECT

                user,

                -- =========================================
                -- BASELINE WORKDAYS
                -- =========================================

                SUM(
                    CASE
                        WHEN day < DATE '{cutoff}'
                         AND is_weekend = 0
                         AND logon_count > 0

                        THEN 1
                        ELSE 0
                    END
                ) AS baseline_workdays,


                -- =========================================
                -- BASELINE CHANNEL ACTIVE DAYS
                -- =========================================

                SUM(
                    CASE
                        WHEN day < DATE '{cutoff}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND external_email_count > 0

                        THEN 1
                        ELSE 0
                    END
                ) AS baseline_email_days,


                SUM(
                    CASE
                        WHEN day < DATE '{cutoff}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND usb_copy_count > 0

                        THEN 1
                        ELSE 0
                    END
                ) AS baseline_usb_days,


                SUM(
                    CASE
                        WHEN day < DATE '{cutoff}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND www_upload_count > 0

                        THEN 1
                        ELSE 0
                    END
                ) AS baseline_http_days,


                -- =========================================
                -- FUTURE WORKDAYS
                -- =========================================

                SUM(
                    CASE
                        WHEN day >= DATE '{cutoff}'
                         AND is_weekend = 0
                         AND logon_count > 0

                        THEN 1
                        ELSE 0
                    END
                ) AS future_workdays,


                -- =========================================
                -- FUTURE CHANNEL ACTIVE DAYS
                -- =========================================

                SUM(
                    CASE
                        WHEN day >= DATE '{cutoff}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND external_email_count > 0

                        THEN 1
                        ELSE 0
                    END
                ) AS future_email_days,


                SUM(
                    CASE
                        WHEN day >= DATE '{cutoff}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND usb_copy_count > 0

                        THEN 1
                        ELSE 0
                    END
                ) AS future_usb_days,


                SUM(
                    CASE
                        WHEN day >= DATE '{cutoff}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND www_upload_count > 0

                        THEN 1
                        ELSE 0
                    END
                ) AS future_http_days


            FROM read_parquet(
                '{user_day_path}'
            )

            WHERE user NOT IN ({malicious_sql})

            GROUP BY user
        )


        SELECT

            COUNT(*) AS benign_users,

            MEDIAN(
                baseline_workdays
            ) AS median_baseline_workdays,

            MEDIAN(
                baseline_email_days
            ) AS median_baseline_email_days,

            MEDIAN(
                baseline_usb_days
            ) AS median_baseline_usb_days,

            MEDIAN(
                baseline_http_days
            ) AS median_baseline_http_days,

            MEDIAN(
                future_workdays
            ) AS median_future_workdays,

            MEDIAN(
                future_email_days
            ) AS median_future_email_days,

            MEDIAN(
                future_usb_days
            ) AS median_future_usb_days,

            MEDIAN(
                future_http_days
            ) AS median_future_http_days

        FROM user_stats

    """).df()


    row = result.iloc[0].to_dict()

    row["cutoff"] = cutoff

    audit_rows.append(row)


con.close()


baseline_window_audit = pd.DataFrame(
    audit_rows
)


baseline_window_audit = baseline_window_audit[
    [
        "cutoff",

        "benign_users",

        "median_baseline_workdays",
        "median_baseline_email_days",
        "median_baseline_usb_days",
        "median_baseline_http_days",

        "median_future_workdays",
        "median_future_email_days",
        "median_future_usb_days",
        "median_future_http_days"
    ]
]


print("=" * 120)
print("BASELINE WINDOW AUDIT")
print("=" * 120)

print(
    baseline_window_audit.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — Baseline window audit calculated."
)

BASELINE WINDOW AUDIT
    cutoff  benign_users  median_baseline_workdays  median_baseline_email_days  median_baseline_usb_days  median_baseline_http_days  median_future_workdays  median_future_email_days  median_future_usb_days  median_future_http_days
2010-07-01        3995.0                     126.0                        80.0                       0.0                        0.0                   230.0                     148.0                     0.0                      0.0
2010-08-01        3995.0                     147.0                        94.0                       0.0                        0.0                   209.0                     135.0                     0.0                      0.0
2010-09-01        3995.0                     169.0                       108.0                       0.0                        0.0                   187.0                     121.0                     0.0                      0.0
2010-10-01        3995.0                     190.0    

In [11]:
# =========================================================
# BASELINE ELIGIBILITY MATRIX
# =========================================================

channel_thresholds = [
    10,
    20,
    30
]

rows = []

con = duckdb.connect()


for cutoff in cutoffs:

    for channel_days in channel_thresholds:

        result = con.execute(f"""

            WITH user_stats AS (

                SELECT

                    user,

                    SUM(
                        CASE
                            WHEN day < DATE '{cutoff}'
                             AND is_weekend = 0
                             AND logon_count > 0
                            THEN 1
                            ELSE 0
                        END
                    ) AS baseline_workdays,


                    SUM(
                        CASE
                            WHEN day < DATE '{cutoff}'
                             AND is_weekend = 0
                             AND logon_count > 0
                             AND external_email_count > 0
                            THEN 1
                            ELSE 0
                        END
                    ) AS email_days,


                    SUM(
                        CASE
                            WHEN day < DATE '{cutoff}'
                             AND is_weekend = 0
                             AND logon_count > 0
                             AND usb_copy_count > 0
                            THEN 1
                            ELSE 0
                        END
                    ) AS usb_days,


                    SUM(
                        CASE
                            WHEN day < DATE '{cutoff}'
                             AND is_weekend = 0
                             AND logon_count > 0
                             AND www_upload_count > 0
                            THEN 1
                            ELSE 0
                        END
                    ) AS http_days,


                    SUM(
                        CASE
                            WHEN day >= DATE '{cutoff}'
                             AND is_weekend = 0
                             AND logon_count > 0
                            THEN 1
                            ELSE 0
                        END
                    ) AS future_workdays


                FROM read_parquet(
                    '{user_day_path}'
                )

                WHERE user NOT IN ({malicious_sql})

                GROUP BY user
            )


            SELECT
                COUNT(*) AS eligible_users

            FROM user_stats

            WHERE baseline_workdays >= 60

              AND email_days >= {channel_days}

              AND usb_days >= {channel_days}

              AND http_days >= {channel_days}

              AND future_workdays >= 30

        """).fetchone()[0]


        rows.append({

            "cutoff":
                cutoff,

            "min_active_days_each_channel":
                channel_days,

            "min_baseline_workdays":
                60,

            "min_future_workdays":
                30,

            "eligible_users":
                result
        })


con.close()


baseline_eligibility = pd.DataFrame(
    rows
)


print("=" * 110)
print("BASELINE ELIGIBILITY MATRIX")
print("=" * 110)

print(
    baseline_eligibility.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — Baseline eligibility matrix calculated."
)

BASELINE ELIGIBILITY MATRIX
    cutoff  min_active_days_each_channel  min_baseline_workdays  min_future_workdays  eligible_users
2010-07-01                            10                     60                   30             248
2010-07-01                            20                     60                   30             176
2010-07-01                            30                     60                   30              86
2010-08-01                            10                     60                   30             257
2010-08-01                            20                     60                   30             193
2010-08-01                            30                     60                   30             122
2010-09-01                            10                     60                   30             260
2010-09-01                            20                     60                   30             207
2010-09-01                            30                     60

In [13]:
# =========================================================
# CALENDAR HORIZON
# =========================================================

dataset_start = pd.Timestamp(
    "2010-01-02"
)

dataset_end = pd.Timestamp(
    "2011-06-01"
)


horizon_rows = []


for cutoff in cutoffs:

    cutoff_date = pd.Timestamp(
        cutoff
    )

    baseline_days = (
        cutoff_date
        - dataset_start
    ).days

    future_days = (
        dataset_end
        - cutoff_date
    ).days + 1


    horizon_rows.append({

        "cutoff":
            cutoff,

        "baseline_calendar_days":
            baseline_days,

        "future_calendar_days":
            future_days
    })


horizon_summary = pd.DataFrame(
    horizon_rows
)


print("=" * 80)
print("BASELINE / FUTURE CALENDAR HORIZON")
print("=" * 80)

print(
    horizon_summary.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — Baseline/future horizons calculated."
)

BASELINE / FUTURE CALENDAR HORIZON
    cutoff  baseline_calendar_days  future_calendar_days
2010-07-01                     180                   336
2010-08-01                     211                   305
2010-09-01                     242                   274
2010-10-01                     272                   244
2010-11-01                     303                   213
2010-12-01                     333                   183
2011-01-01                     364                   152
2011-02-01                     395                   121

✅ SUCCESS — Baseline/future horizons calculated.


In [15]:
# =========================================================
# FINAL BASELINE-STRENGTH CHECK
# Fixed cutoff: 2011-01-01
# =========================================================

FINAL_CUTOFF = "2011-01-01"

channel_thresholds = [
    20,
    30,
    40,
    50
]

rows = []

con = duckdb.connect()

for channel_days in channel_thresholds:

    eligible = con.execute(f"""

        WITH user_stats AS (

            SELECT
                user,

                SUM(
                    CASE
                        WHEN day < DATE '{FINAL_CUTOFF}'
                         AND is_weekend = 0
                         AND logon_count > 0
                        THEN 1 ELSE 0
                    END
                ) AS baseline_workdays,

                SUM(
                    CASE
                        WHEN day < DATE '{FINAL_CUTOFF}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND external_email_count > 0
                        THEN 1 ELSE 0
                    END
                ) AS email_days,

                SUM(
                    CASE
                        WHEN day < DATE '{FINAL_CUTOFF}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND usb_copy_count > 0
                        THEN 1 ELSE 0
                    END
                ) AS usb_days,

                SUM(
                    CASE
                        WHEN day < DATE '{FINAL_CUTOFF}'
                         AND is_weekend = 0
                         AND logon_count > 0
                         AND www_upload_count > 0
                        THEN 1 ELSE 0
                    END
                ) AS http_days,

                SUM(
                    CASE
                        WHEN day >= DATE '{FINAL_CUTOFF}'
                         AND is_weekend = 0
                         AND logon_count > 0
                        THEN 1 ELSE 0
                    END
                ) AS future_workdays

            FROM read_parquet(
                '{user_day_path}'
            )

            WHERE user NOT IN ({malicious_sql})

            GROUP BY user
        )

        SELECT COUNT(*)

        FROM user_stats

        WHERE baseline_workdays >= 60

          AND email_days >= {channel_days}
          AND usb_days >= {channel_days}
          AND http_days >= {channel_days}

          AND future_workdays >= 30

    """).fetchone()[0]

    rows.append({
        "min_active_days_each_channel": channel_days,
        "eligible_users": eligible
    })

con.close()

final_strength_check = pd.DataFrame(rows)

print("=" * 70)
print("FINAL BASELINE STRENGTH CHECK")
print("Cutoff: 2011-01-01")
print("=" * 70)

print(
    final_strength_check.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — Final baseline-strength check calculated."
)

FINAL BASELINE STRENGTH CHECK
Cutoff: 2011-01-01
 min_active_days_each_channel  eligible_users
                           20             230
                           30             199
                           40             158
                           50             126

✅ SUCCESS — Final baseline-strength check calculated.


In [17]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

PROCESSED = PROJECT_ROOT / "data" / "processed"
FEATURES = PROJECT_ROOT / "data" / "features"

FEATURES.mkdir(parents=True, exist_ok=True)

USER_DAY = PROCESSED / "user_day_r62_v1.parquet"

BASELINE_FILE = FEATURES / "user_baseline_r62_v1.parquet"

user_day_path = str(USER_DAY).replace("\\", "/")
baseline_path = str(BASELINE_FILE).replace("\\", "/")

BASELINE_CUTOFF = "2011-01-01"

malicious_users = [
    "ACM2278",
    "CMP2946",
    "PLJ1771",
    "CDE1846",
    "MBG3183"
]

malicious_sql = ", ".join(
    f"'{u}'"
    for u in malicious_users
)

assert USER_DAY.exists()

print("Input :", USER_DAY)
print("Output:", BASELINE_FILE)
print("Cutoff:", BASELINE_CUTOFF)

print("\n✅ SUCCESS — Baseline setup ready.")

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
Input : C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\user_day_r62_v1.parquet
Output: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\user_baseline_r62_v1.parquet
Cutoff: 2011-01-01

✅ SUCCESS — Baseline setup ready.


In [19]:
# =========================================================
# BUILD USER BASELINE PROFILES
# =========================================================

con = duckdb.connect()

con.execute(f"""
COPY (

WITH

-- =========================================================
-- ALL BENIGN USERS
-- =========================================================

benign_users AS (

    SELECT DISTINCT user

    FROM read_parquet(
        '{user_day_path}'
    )

    WHERE user NOT IN ({malicious_sql})
),


-- =========================================================
-- BASELINE WORKDAYS ONLY
-- =========================================================

baseline_days AS (

    SELECT *

    FROM read_parquet(
        '{user_day_path}'
    )

    WHERE day < DATE '{BASELINE_CUTOFF}'

      AND is_weekend = 0

      AND logon_count > 0

      AND user NOT IN ({malicious_sql})
),


-- =========================================================
-- CORE WORKDAY COUNTS
-- =========================================================

core AS (

    SELECT
        user,

        COUNT(*) AS baseline_workdays,

        MIN(day) AS baseline_first_day,

        MAX(day) AS baseline_last_day

    FROM baseline_days

    GROUP BY user
),


-- =========================================================
-- EMAIL ACTIVE-DAY STATS
-- =========================================================

email_stats AS (

    SELECT
        user,

        COUNT(*) AS email_active_days,

        AVG(external_email_count)
            AS email_count_mean_active,

        QUANTILE_CONT(
            external_email_count, 0.50
        ) AS email_count_p50,

        QUANTILE_CONT(
            external_email_count, 0.75
        ) AS email_count_p75,

        QUANTILE_CONT(
            external_email_count, 0.90
        ) AS email_count_p90,

        QUANTILE_CONT(
            external_email_count, 0.95
        ) AS email_count_p95,


        AVG(external_email_bytes)
            AS email_bytes_mean_active,

        QUANTILE_CONT(
            external_email_bytes, 0.50
        ) AS email_bytes_p50,

        QUANTILE_CONT(
            external_email_bytes, 0.75
        ) AS email_bytes_p75,

        QUANTILE_CONT(
            external_email_bytes, 0.90
        ) AS email_bytes_p90,

        QUANTILE_CONT(
            external_email_bytes, 0.95
        ) AS email_bytes_p95

    FROM baseline_days

    WHERE external_email_count > 0

    GROUP BY user
),


email_mad AS (

    SELECT
        b.user,

        MEDIAN(
            ABS(
                b.external_email_count
                - e.email_count_p50
            )
        ) AS email_count_mad,

        MEDIAN(
            ABS(
                b.external_email_bytes
                - e.email_bytes_p50
            )
        ) AS email_bytes_mad

    FROM baseline_days b

    INNER JOIN email_stats e
        ON b.user = e.user

    WHERE b.external_email_count > 0

    GROUP BY b.user
),


-- =========================================================
-- USB ACTIVE-DAY STATS
-- =========================================================

usb_stats AS (

    SELECT
        user,

        COUNT(*) AS usb_active_days,

        AVG(usb_copy_count)
            AS usb_copy_mean_active,

        QUANTILE_CONT(
            usb_copy_count, 0.50
        ) AS usb_copy_p50,

        QUANTILE_CONT(
            usb_copy_count, 0.75
        ) AS usb_copy_p75,

        QUANTILE_CONT(
            usb_copy_count, 0.90
        ) AS usb_copy_p90,

        QUANTILE_CONT(
            usb_copy_count, 0.95
        ) AS usb_copy_p95

    FROM baseline_days

    WHERE usb_copy_count > 0

    GROUP BY user
),


usb_mad AS (

    SELECT
        b.user,

        MEDIAN(
            ABS(
                b.usb_copy_count
                - u.usb_copy_p50
            )
        ) AS usb_copy_mad

    FROM baseline_days b

    INNER JOIN usb_stats u
        ON b.user = u.user

    WHERE b.usb_copy_count > 0

    GROUP BY b.user
),


-- =========================================================
-- HTTP UPLOAD ACTIVE-DAY STATS
-- =========================================================

http_stats AS (

    SELECT
        user,

        COUNT(*) AS http_upload_active_days,

        AVG(www_upload_count)
            AS http_upload_mean_active,

        QUANTILE_CONT(
            www_upload_count, 0.50
        ) AS http_upload_p50,

        QUANTILE_CONT(
            www_upload_count, 0.75
        ) AS http_upload_p75,

        QUANTILE_CONT(
            www_upload_count, 0.90
        ) AS http_upload_p90,

        QUANTILE_CONT(
            www_upload_count, 0.95
        ) AS http_upload_p95

    FROM baseline_days

    WHERE www_upload_count > 0

    GROUP BY user
),


http_mad AS (

    SELECT
        b.user,

        MEDIAN(
            ABS(
                b.www_upload_count
                - h.http_upload_p50
            )
        ) AS http_upload_mad

    FROM baseline_days b

    INNER JOIN http_stats h
        ON b.user = h.user

    WHERE b.www_upload_count > 0

    GROUP BY b.user
)


-- =========================================================
-- FINAL USER BASELINE
-- =========================================================

SELECT

    bu.user,

    DATE '2010-01-02'
        AS baseline_period_start,

    DATE '2010-12-31'
        AS baseline_period_end,

    COALESCE(
        c.baseline_workdays,
        0
    ) AS baseline_workdays,

    c.baseline_first_day,

    c.baseline_last_day,


    -- EMAIL
    COALESCE(
        e.email_active_days,
        0
    ) AS email_active_days,

    CASE
        WHEN COALESCE(
            c.baseline_workdays,
            0
        ) > 0

        THEN
            COALESCE(
                e.email_active_days,
                0
            )::DOUBLE
            /
            c.baseline_workdays

        ELSE 0
    END AS email_active_rate,

    e.email_count_mean_active,
    e.email_count_p50,
    e.email_count_p75,
    e.email_count_p90,
    e.email_count_p95,

    em.email_count_mad,

    e.email_bytes_mean_active,
    e.email_bytes_p50,
    e.email_bytes_p75,
    e.email_bytes_p90,
    e.email_bytes_p95,

    em.email_bytes_mad,


    -- USB
    COALESCE(
        u.usb_active_days,
        0
    ) AS usb_active_days,

    CASE
        WHEN COALESCE(
            c.baseline_workdays,
            0
        ) > 0

        THEN
            COALESCE(
                u.usb_active_days,
                0
            )::DOUBLE
            /
            c.baseline_workdays

        ELSE 0
    END AS usb_active_rate,

    u.usb_copy_mean_active,
    u.usb_copy_p50,
    u.usb_copy_p75,
    u.usb_copy_p90,
    u.usb_copy_p95,

    um.usb_copy_mad,


    -- HTTP UPLOAD
    COALESCE(
        h.http_upload_active_days,
        0
    ) AS http_upload_active_days,

    CASE
        WHEN COALESCE(
            c.baseline_workdays,
            0
        ) > 0

        THEN
            COALESCE(
                h.http_upload_active_days,
                0
            )::DOUBLE
            /
            c.baseline_workdays

        ELSE 0
    END AS http_upload_active_rate,

    h.http_upload_mean_active,
    h.http_upload_p50,
    h.http_upload_p75,
    h.http_upload_p90,
    h.http_upload_p95,

    hm.http_upload_mad


FROM benign_users bu

LEFT JOIN core c
    ON bu.user = c.user

LEFT JOIN email_stats e
    ON bu.user = e.user

LEFT JOIN email_mad em
    ON bu.user = em.user

LEFT JOIN usb_stats u
    ON bu.user = u.user

LEFT JOIN usb_mad um
    ON bu.user = um.user

LEFT JOIN http_stats h
    ON bu.user = h.user

LEFT JOIN http_mad hm
    ON bu.user = hm.user

ORDER BY bu.user

)

TO '{baseline_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert BASELINE_FILE.exists()

print("Created:")
print(BASELINE_FILE)

print(
    "\n✅ SUCCESS — user_baseline_r62_v1.parquet created."
)

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\user_baseline_r62_v1.parquet

✅ SUCCESS — user_baseline_r62_v1.parquet created.


In [21]:
# =========================================================
# INSPECT BASELINE
# =========================================================

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{baseline_path}'
)
""").df()

sample = con.execute(f"""
SELECT *
FROM read_parquet(
    '{baseline_path}'
)
ORDER BY user
LIMIT 10
""").df()

con.close()

print("=" * 100)
print("BASELINE SCHEMA")
print("=" * 100)

print(
    schema[
        ["column_name", "column_type"]
    ].to_string(index=False)
)

print("\nTotal columns:", len(schema))

print("\n" + "=" * 100)
print("SAMPLE")
print("=" * 100)

print(
    sample.to_string(index=False)
)

print(
    "\n✅ SUCCESS — Baseline profile inspected."
)

BASELINE SCHEMA
            column_name column_type
                   user     VARCHAR
  baseline_period_start        DATE
    baseline_period_end        DATE
      baseline_workdays      BIGINT
     baseline_first_day        DATE
      baseline_last_day        DATE
      email_active_days      BIGINT
      email_active_rate      DOUBLE
email_count_mean_active      DOUBLE
        email_count_p50      DOUBLE
        email_count_p75      DOUBLE
        email_count_p90      DOUBLE
        email_count_p95      DOUBLE
        email_count_mad      DOUBLE
email_bytes_mean_active      DOUBLE
        email_bytes_p50      DOUBLE
        email_bytes_p75      DOUBLE
        email_bytes_p90      DOUBLE
        email_bytes_p95      DOUBLE
        email_bytes_mad      DOUBLE
        usb_active_days      BIGINT
        usb_active_rate      DOUBLE
   usb_copy_mean_active      DOUBLE
           usb_copy_p50      DOUBLE
           usb_copy_p75      DOUBLE
           usb_copy_p90      DOUBLE
           u

In [23]:
# =========================================================
# BASELINE CORE VALIDATION
# =========================================================

con = duckdb.connect()

summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,

    COUNT(
        DISTINCT user
    ) AS users,

    MIN(baseline_workdays)
        AS min_workdays,

    MEDIAN(baseline_workdays)
        AS median_workdays,

    MAX(baseline_workdays)
        AS max_workdays

FROM read_parquet(
    '{baseline_path}'
)
""").df()


duplicates = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT
        user,
        COUNT(*) AS n

    FROM read_parquet(
        '{baseline_path}'
    )

    GROUP BY user

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


malicious_present = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{baseline_path}'
)

WHERE user IN ({malicious_sql})
""").fetchone()[0]


bad_active_days = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{baseline_path}'
)

WHERE email_active_days > baseline_workdays

   OR usb_active_days > baseline_workdays

   OR http_upload_active_days > baseline_workdays
""").fetchone()[0]


bad_rates = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{baseline_path}'
)

WHERE email_active_rate NOT BETWEEN 0 AND 1

   OR usb_active_rate NOT BETWEEN 0 AND 1

   OR http_upload_active_rate NOT BETWEEN 0 AND 1
""").fetchone()[0]

con.close()


print("=" * 100)
print("BASELINE CORE VALIDATION")
print("=" * 100)

print(
    summary.to_string(
        index=False
    )
)

print("\nDuplicate users       :", duplicates)
print("Malicious users found :", malicious_present)
print("Active days > workdays:", bad_active_days)
print("Invalid active rates  :", bad_rates)


assert int(summary.loc[0, "rows"]) == 3995
assert int(summary.loc[0, "users"]) == 3995

assert duplicates == 0
assert malicious_present == 0
assert bad_active_days == 0
assert bad_rates == 0


print(
    "\n✅ SUCCESS — BASELINE CORE VALIDATION PASSED."
)

BASELINE CORE VALIDATION
 rows  users  min_workdays  median_workdays  max_workdays
 3995   3995            21            251.0           260

Duplicate users       : 0
Malicious users found : 0
Active days > workdays: 0
Invalid active rates  : 0

✅ SUCCESS — BASELINE CORE VALIDATION PASSED.


In [25]:
# =========================================================
# BASELINE PERCENTILE SANITY
# =========================================================

con = duckdb.connect()

bad_percentiles = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{baseline_path}'
)

WHERE

    (
        email_active_days > 0

        AND NOT (
            email_count_p50
            <= email_count_p75

            AND email_count_p75
            <= email_count_p90

            AND email_count_p90
            <= email_count_p95
        )
    )

    OR

    (
        usb_active_days > 0

        AND NOT (
            usb_copy_p50
            <= usb_copy_p75

            AND usb_copy_p75
            <= usb_copy_p90

            AND usb_copy_p90
            <= usb_copy_p95
        )
    )

    OR

    (
        http_upload_active_days > 0

        AND NOT (
            http_upload_p50
            <= http_upload_p75

            AND http_upload_p75
            <= http_upload_p90

            AND http_upload_p90
            <= http_upload_p95
        )
    )
""").fetchone()[0]

con.close()


print(
    "Users with invalid percentile ordering:",
    bad_percentiles
)

assert bad_percentiles == 0

print(
    "\n✅ SUCCESS — BASELINE PERCENTILES ARE VALID."
)

Users with invalid percentile ordering: 0

✅ SUCCESS — BASELINE PERCENTILES ARE VALID.


In [27]:
# =========================================================
# BASELINE COVERAGE SUMMARY
# =========================================================

con = duckdb.connect()

coverage = con.execute(f"""
SELECT

    COUNT(*) AS benign_users,

    SUM(
        CASE
            WHEN email_active_days > 0
            THEN 1 ELSE 0
        END
    ) AS users_with_email,

    SUM(
        CASE
            WHEN usb_active_days > 0
            THEN 1 ELSE 0
        END
    ) AS users_with_usb,

    SUM(
        CASE
            WHEN http_upload_active_days > 0
            THEN 1 ELSE 0
        END
    ) AS users_with_http_upload,

    SUM(
        CASE
            WHEN email_active_days >= 50
             AND usb_active_days >= 50
             AND http_upload_active_days >= 50
            THEN 1 ELSE 0
        END
    ) AS strict_three_channel_users

FROM read_parquet(
    '{baseline_path}'
)
""").df()


active_summary = con.execute(f"""
SELECT

    MEDIAN(email_active_days)
        AS median_email_days,

    MEDIAN(usb_active_days)
        AS median_usb_days,

    MEDIAN(http_upload_active_days)
        AS median_http_days

FROM read_parquet(
    '{baseline_path}'
)
""").df()

con.close()


print("=" * 100)
print("BASELINE COVERAGE")
print("=" * 100)

print(
    coverage.to_string(
        index=False
    )
)

print("\nActive-day medians across all benign users:")

print(
    active_summary.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — Baseline coverage summary calculated."
)

BASELINE COVERAGE
 benign_users  users_with_email  users_with_usb  users_with_http_upload  strict_three_channel_users
         3995            3994.0           783.0                   615.0                       131.0

Active-day medians across all benign users:
 median_email_days  median_usb_days  median_http_days
             160.0              0.0               0.0

✅ SUCCESS — Baseline coverage summary calculated.
